In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv(
    "data/Indian_ODI_Cricketers_With_Role.csv"
)

print(df.shape)

df.head()

(250, 20)


,No,Name,First,Last,Mat,Inn,NO.1,Runs,HS,Avg,Balls,Mdn,Runs_1,Wkt,BBM,Avg_2,Ca,St,Selected,Role
0,1,Syed Abid Ali,1974,1975,5,3.0,0.0,93.0,70,31.00,336.0,10.0,187.0,7.0,2/22,26.71,0,0,0,Batsman
1,2,Bishan Singh Bedi,1974,1979,10,7.0,2.0,31.0,13,6.20,590.0,17.0,340.0,7.0,2/44,48.57,4,0,0,Batsman
2,3,Farokh Engineer,1974,1975,5,4.0,1.0,114.0,54*,38.00,0.0,0.0,0.0,0.0,NaN,0.00,3,1,0,Wicketkeeper
3,4,Sunil Gavaskar,1974,1987,108,102.0,14.0,3092.0,103*,35.13,20.0,0.0,25.0,1.0,1/10,25.00,22,0,0,Batsman
4,5,Madan Lal,1974,1987,67,35.0,14.0,401.0,53*,19.09,3164.0,44.0,2137.0,73.0,4/20,29.27,18,0,0,Bowler


In [3]:
features = [
    'Avg',
    'Runs',
    'strike_rate',
    'consistency',
    'Wkt',
    'Avg_2',
    'economy',
    'Mat',
    'Mdn'
]

In [4]:
def generate_role_predictions(role_name):

    role_df = df[
        df['Role'] == role_name
    ].copy()

    print(
        f"\nPlayers in {role_name}:",
        len(role_df)
    )

    X = role_df[features]

    y = role_df['Selected']

    scaler = MinMaxScaler()

    X_scaled = scaler.fit_transform(X)

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(
        X_scaled,
        y
    )

    role_df[
        'selection_probability'
    ] = model.predict_proba(
        X_scaled
    )[:,1]

    role_df = role_df.sort_values(
        by='selection_probability',
        ascending=False
    )

    return role_df

In [5]:
best_batsmen = generate_role_predictions(
    "Batsman"
)

best_batsmen[
    [
        'Name',
        'Role',
        'selection_probability'
    ]
].head(5)


Players in Batsman: 181


KeyError: "['strike_rate', 'consistency', 'economy'] not in index"

In [6]:
print(df.columns.tolist())

['No', 'Name', 'First', 'Last', 'Mat', 'Inn', 'NO.1', 'Runs', 'HS', 'Avg', 'Balls', 'Mdn', 'Runs_1', 'Wkt', 'BBM', 'Avg_2', 'Ca', 'St', 'Selected', 'Role']


In [7]:
# Avoid division by zero

df['Balls'] = df['Balls'].replace(0, 1)
df['Inn'] = df['Inn'].replace(0, 1)

# Create Features

df['strike_rate'] = (
    df['Runs'] / df['Balls']
) * 100

df['consistency'] = (
    df['Runs'] / df['Inn']
)

df['economy'] = (
    df['Runs_1'] / df['Balls']
) * 6

print(
    df[
        [
            'strike_rate',
            'consistency',
            'economy'
        ]
    ].head()
)

    strike_rate  consistency   economy
0     27.678571    31.000000  3.339286
1      5.254237     4.428571  3.457627
2  11400.000000    28.500000  0.000000
3  15460.000000    30.313725  7.500000
4     12.673831    11.457143  4.052465


In [8]:
best_batsmen = generate_role_predictions(
    "Batsman"
)


Players in Batsman: 181


In [9]:
print(df['Role'].unique())

<StringArray>
['Batsman', 'Wicketkeeper', 'Bowler', 'AllRounder']
Length: 4, dtype: str


In [10]:
best_batsmen = generate_role_predictions(
    "Batsman"
)

best_batsmen[
    [
        'Name',
        'Role',
        'selection_probability'
    ]
].head(5)


Players in Batsman: 181


,Name,Role,selection_probability
174,Virat Kohli,Batsman,0.885
167,Rohit Sharma,Batsman,0.850
187,Shikhar Dhawan,Batsman,0.825
226,Shubman Gill,Batsman,0.815
218,Shreyas Iyer,Batsman,0.750


In [11]:
best_bowlers = generate_role_predictions(
    "Bowler"
)

best_bowlers[
    [
        'Name',
        'Role',
        'selection_probability'
    ]
].head(5)


Players in Bowler: 27


,Name,Role,selection_probability
184,Ravichandran Ashwin,Bowler,0.735
193,Bhuvneshwar Kumar,Bowler,0.730
209,Jasprit Bumrah,Bowler,0.690
194,Mohammed Shami,Bowler,0.645
136,Ashish Nehra,Bowler,0.190


In [12]:
best_allrounders = generate_role_predictions(
    "AllRounder"
)

best_allrounders[
    [
        'Name',
        'Role',
        'selection_probability'
    ]
].head(3)


Players in AllRounder: 16


,Name,Role,selection_probability
176,Ravindra Jadeja,AllRounder,0.650
214,Hardik Pandya,AllRounder,0.645
171,Yusuf Pathan,AllRounder,0.140


In [13]:
best_wicketkeepers = generate_role_predictions(
    "Wicketkeeper"
)

best_wicketkeepers[
    [
        'Name',
        'Role',
        'selection_probability'
    ]
].head(2)


Players in Wicketkeeper: 26


,Name,Role,selection_probability
240,Sanju Samson,Wicketkeeper,0.690
212,KL Rahul,Wicketkeeper,0.655


In [14]:
final_squad = pd.concat([
    best_batsmen.head(5),
    best_wicketkeepers.head(2),
    best_allrounders.head(3),
    best_bowlers.head(5)
])

final_squad = final_squad.sort_values(
    by='selection_probability',
    ascending=False
)

final_squad[
    [
        'Name',
        'Role',
        'selection_probability'
    ]
]

,Name,Role,selection_probability
174,Virat Kohli,Batsman,0.885
167,Rohit Sharma,Batsman,0.850
187,Shikhar Dhawan,Batsman,0.825
226,Shubman Gill,Batsman,0.815
218,Shreyas Iyer,Batsman,0.750
184,Ravichandran Ashwin,Bowler,0.735
193,Bhuvneshwar Kumar,Bowler,0.730
209,Jasprit Bumrah,Bowler,0.690
240,Sanju Samson,Wicketkeeper,0.690
212,KL Rahul,Wicketkeeper,0.655
